# DeepFakeGuard-Text — Revision Experiments (3ICT2026, paper #1571340300)

Run in **Colab (GPU, T4 is fine)**. Put `reddit_filtered_dataset.csv` and `AIGTxt.csv` in `/content/`. **Tip:** mount Google Drive and set `CFG['save_dir']` to a Drive folder (in the first code cell) so a disconnect doesn't lose finished runs; finished runs are skipped when you re-run.

Run the cells **top to bottom**. Steps A–E, H, I, J are the must-haves; F and G strengthen the paper if time allows.

| Step | What it does | Reviewer point |
|---|---|---|
| A | Counts after every preprocessing step, duplicate removal, group-aware split, near-duplicate leakage report, released split IDs | R1, R2 |
| B | TF-IDF baseline; SMOTE vs. none / class-weights / random oversampling on frozen embeddings | R1–R4 |
| C | Hyper-parameter search on train/val only (no test contact) | R1, R2 |
| D | Fine-tuned ablation: none / class-weight / random oversampling / latent SMOTE, 3 seeds, same splits | all |
| E | mean ± std, bootstrap 95% CIs, McNemar, raw confusion counts, threshold sweep | R1, R2, R3 |
| F | 5-fold group-aware CV with no checkpoint selection (replaces Table II) | R1, R2 |
| G | Cross-source generalisation (AIGTxt↔Reddit) | R1–R4 |
| H | Parameter count + latency benchmark (batch sizes, FP32/FP16, GPU/CPU, percentiles) | R1–R3 |
| I | Figures at 300 dpi (error bars, selected config marked, raw counts) | R1, R4 |
| J | Exports `RESULTS_FOR_PAPER.txt` + zip to send back | — |

**Short on time?** Right before Step C set `CFG['seeds'] = [42, 43]`, use `seeds=(42,)` in Step C, and set `CFG['cv_variants'] = ['smote_nn']`. To add baselines, set `CFG['models'] = ['distilbert-base-uncased', 'roberta-base']` and re-run Step D (CV, figures and cross-source use the first model).

In [ ]:
!pip install -q transformers imbalanced-learn scikit-learn pandas numpy scipy tqdm

## 1. Configuration, data loading, splits, metrics

In [ ]:
import os, json, time, copy, random, warnings, hashlib, io, contextlib
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             precision_recall_fscore_support, confusion_matrix)
from scipy.stats import binomtest

CFG = dict(
    reddit_csv='/content/reddit_filtered_dataset.csv',
    aig_csv='/content/AIGTxt.csv',
    save_dir='/content/deepfakeguard_v2',
    seeds=[42, 43, 44],
    epochs=4, lr=3e-5, batch_size=16, dropout=0.3, max_len=256, weight_decay=0.01,
    models=['distilbert-base-uncased'],          # add 'roberta-base' / 'bert-base-uncased' for baselines
    variants=['none', 'classweight', 'ros', 'smote_nn'],
    cv_variants=['none', 'smote_nn'],            # variants for the 5-fold CV (use ['smote_nn'] only if short on time)
)
os.makedirs(CFG['save_dir'], exist_ok=True)
os.makedirs(CFG['save_dir'] + '/preds', exist_ok=True)

def load_data(cfg):
    rd = pd.read_csv(cfg['reddit_csv'])[['Data', 'Labels']].rename(columns={'Data': 'text', 'Labels': 'label'})
    rd['source'] = 'reddit'
    rd['group'] = ['reddit_%d' % i for i in range(len(rd))]
    aig = pd.read_csv(cfg['aig_csv'])
    # A human paragraph and its ChatGPT rewrite share one AIGTxt row -> same group
    h = pd.DataFrame({'text': aig['Human-Generated'], 'label': 0, 'source': 'aigtxt'})
    h['group'] = ['aig_%d' % i for i in aig.index]
    a = pd.DataFrame({'text': aig['ChatGPT-Generated'], 'label': 1, 'source': 'aigtxt'})
    a['group'] = ['aig_%d' % i for i in aig.index]
    df = pd.concat([rd, h, a], ignore_index=True)

    log = []
    def snap(step, d):
        log.append(dict(step=step, total=len(d), human=int((d.label == 0).sum()), ai=int((d.label == 1).sum())))
    snap('raw rows', df)
    df = df.dropna(subset=['text']).copy()
    df['text'] = df['text'].astype(str).str.strip()
    snap('after dropna', df)
    df = df[df['text'].str.len() > 10].copy()
    snap('after length > 10 chars', df)
    df['norm'] = df['text'].str.lower().str.replace(r'\s+', ' ', regex=True)
    n_lab = df.groupby('norm')['label'].nunique()
    conflict = set(n_lab[n_lab > 1].index)
    df = df[~df['norm'].isin(conflict)]
    snap('after dropping label-conflicting duplicates', df)
    df = df.drop_duplicates('norm').reset_index(drop=True)
    snap('after exact-duplicate removal', df)
    return df, pd.DataFrame(log)

def make_splits(df, seed=42, test_folds=5, val_folds=8):
    """Group-aware, stratified train/val/test. Texts that share a group (same AIGTxt row) never straddle splits."""
    y, g = df['label'].values, df['group'].values
    idx = np.arange(len(df))
    sg = StratifiedGroupKFold(n_splits=test_folds, shuffle=True, random_state=seed)
    dev, test = next(iter(sg.split(idx, y, g)))
    sg2 = StratifiedGroupKFold(n_splits=val_folds, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(iter(sg2.split(dev, y[dev], g[dev])))
    tr, va = dev[tr_rel], dev[va_rel]
    assert not (set(g[tr]) & set(g[va])) and not (set(g[tr]) & set(g[test])) and not (set(g[va]) & set(g[test]))
    return tr, va, test

def leakage_report(df, tr, te, thresholds=(0.8, 0.9, 0.95)):
    """Fraction of test texts that have a near-duplicate (TF-IDF cosine) in the training set."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=100000)
    Xtr = vec.fit_transform(df['text'].iloc[tr]); Xte = vec.transform(df['text'].iloc[te])
    mx = np.zeros(Xte.shape[0])
    for s in range(0, Xte.shape[0], 500):
        mx[s:s + 500] = (Xte[s:s + 500] @ Xtr.T).max(axis=1).toarray().ravel()
    rows = {f'>= {t}': float((mx >= t).mean()) for t in thresholds}
    print('Share of test texts with a train near-duplicate (TF-IDF cosine):', rows)
    return mx

def metrics_from(y, p, thr=0.5):
    y = np.asarray(y); p = np.asarray(p); pred = (p >= thr).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(acc=accuracy_score(y, pred), macro_f1=f1_score(y, pred, average='macro'),
                weighted_f1=f1_score(y, pred, average='weighted'), auc=roc_auc_score(y, p),
                bal_acc=float(rc.mean()), human_recall=rc[0], ai_recall=rc[1],
                human_precision=pr[0], ai_precision=pr[1],
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

def bootstrap_ci(y, p, fn, n=2000, seed=0):
    y = np.asarray(y); p = np.asarray(p); rng = np.random.RandomState(seed); v = []
    for _ in range(n):
        i = rng.randint(0, len(y), len(y))
        if len(np.unique(y[i])) < 2: continue
        v.append(fn(y[i], p[i]))
    return tuple(np.percentile(v, [2.5, 97.5]))

def mcnemar_exact(y, pred1, pred2):
    c1, c2 = (pred1 == y), (pred2 == y)
    b, c = int((c1 & ~c2).sum()), int((~c1 & c2).sum()); n = b + c
    return b, c, (binomtest(b, n, 0.5).pvalue if n > 0 else 1.0)

REPORT = []   # everything that goes into RESULTS_FOR_PAPER.txt

def capture_section(title, fn, *a, **k):
    """Run fn, print what it prints (and its returned DataFrame), and store it for the results pack."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        out = fn(*a, **k)
    txt = buf.getvalue()
    if isinstance(out, (pd.DataFrame, pd.Series)):
        txt += out.to_string()
    print(f"\n===== {title} =====\n{txt}")
    REPORT.append(f"\n===== {title} =====\n{txt}")
    return out


## 2. Balancing strategies (SMOTE variant + controls)

In [ ]:
from imblearn.over_sampling import SMOTE

def build_train_indices(variant, tr_idx, y, emb=None, seed=42, k=5):
    """Return (indices into the dataset, possibly with repeats, info dict).
    'none' / 'classweight': real samples only.   'ros': duplicate random minority texts.
    'smote_nn': SMOTE on frozen-encoder embeddings; each synthetic vector is mapped to its nearest REAL
                minority text (cosine), because a synthetic vector has no text the encoder can be fine-tuned on."""
    tr_idx = np.asarray(tr_idx); ytr = y[tr_idx]; rng = np.random.RandomState(seed)
    n0, n1 = int((ytr == 0).sum()), int((ytr == 1).sum())
    minority = 1 if n1 < n0 else 0
    info = dict(variant=variant, n_real=len(tr_idx), n_extra=0)
    if variant in ('none', 'classweight') or n0 == n1:
        return tr_idx, info
    if variant == 'ros':
        extra = rng.choice(tr_idx[ytr == minority], abs(n0 - n1), replace=True)
    elif variant == 'smote_nn':
        Xtr = emb[tr_idx]
        Xr, yr = SMOTE(k_neighbors=k, random_state=seed).fit_resample(Xtr, ytr)
        Xs = Xr[len(tr_idx):]                                  # synthetic rows are appended after the originals
        assert len(Xs) == abs(n0 - n1) and (yr[len(tr_idx):] == minority).all()
        pool_pos = np.where(ytr == minority)[0]
        B = Xtr[pool_pos]; B = B / np.linalg.norm(B, axis=1, keepdims=True)
        A = Xs / np.linalg.norm(Xs, axis=1, keepdims=True)
        nn_pos = np.concatenate([(A[s:s + 2048] @ B.T).argmax(1) for s in range(0, len(A), 2048)])
        extra = tr_idx[pool_pos[nn_pos]]
        info['unique_source_texts'] = int(len(np.unique(extra)))
        info['pct_synthetic_that_duplicate_an_already_used_text'] = float(100 * (1 - len(np.unique(extra)) / len(extra)))
    else:
        raise ValueError(variant)
    info['n_extra'] = int(len(extra))
    return np.concatenate([tr_idx, extra]), info


## 3. Model, training and evaluation code

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print('Device:', DEVICE, '| torch', torch.__version__, '| AMP fp16 training:', USE_AMP)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def T(a):
    return torch.as_tensor(np.asarray(a), dtype=torch.long)

class Detector(nn.Module):
    """Encoder [CLS] -> Dropout(p) -> FC(h,256) -> ReLU -> Dropout(p/2) -> FC(256,2). ALL encoder weights are fine-tuned."""
    def __init__(self, name, p=0.3):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        h = self.enc.config.hidden_size
        self.drop1, self.fc = nn.Dropout(p), nn.Linear(h, 256)
        self.drop2, self.out = nn.Dropout(p / 2), nn.Linear(256, 2)
    def forward(self, ids, mask):
        cls = self.enc(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0]
        f = F.relu(self.fc(self.drop1(cls)))
        return self.out(self.drop2(f)), f

_D_CACHE = {}
def get_D(name, df, need_emb=False):
    """Tokenised tensors (+ frozen-encoder CLS embeddings, only needed for SMOTE) for one backbone."""
    if name not in _D_CACHE:
        tok = AutoTokenizer.from_pretrained(name)
        enc = tok(list(df['text']), max_length=CFG['max_len'], padding='max_length', truncation=True, return_tensors='pt')
        _D_CACHE[name] = dict(tok=tok, ids=enc['input_ids'], mask=enc['attention_mask'], y=df['label'].values)
    D = _D_CACHE[name]
    if need_emb and 'emb' not in D:
        h = hashlib.md5(''.join(df['norm']).encode()).hexdigest()[:10]
        path = f"{CFG['save_dir']}/emb_{name.replace('/', '_')}_{h}.npy"
        if os.path.exists(path):
            D['emb'] = np.load(path)
        else:
            enc_model = AutoModel.from_pretrained(name).to(DEVICE).eval(); out = []
            with torch.no_grad():
                for s in tqdm(range(0, len(D['ids']), 128), desc='frozen embeddings'):
                    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                        hs = enc_model(input_ids=D['ids'][s:s + 128].to(DEVICE),
                                       attention_mask=D['mask'][s:s + 128].to(DEVICE)).last_hidden_state[:, 0]
                    out.append(hs.float().cpu().numpy())
            del enc_model; torch.cuda.empty_cache()
            D['emb'] = np.vstack(out); np.save(path, D['emb'])
    return D

@torch.no_grad()
def predict_probs(model, ids, mask, bs=64):
    model.eval(); P = []
    for s in range(0, len(ids), bs):
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            logits, _ = model(ids[s:s + bs].to(DEVICE), mask[s:s + bs].to(DEVICE))
        P.append(F.softmax(logits.float(), 1)[:, 1].cpu().numpy())
    return np.concatenate(P)

def run_experiment(name, variant, seed, D, split, cfg, save_path=None, eval_test=True):
    """Fine-tune end-to-end.
    va given  -> checkpoint chosen on the VALIDATION split only.   va None -> last epoch is used (no selection).
    The test split is scored once at the end (skipped when eval_test=False, e.g. for the hyper-parameter search)."""
    tr, va, te = split; y = D['y']; t_start = time.time()
    set_seed(seed)
    idx, info = build_train_indices(variant, tr, y, D.get('emb'), seed)
    ti = T(idx)
    loader = DataLoader(TensorDataset(D['ids'][ti], D['mask'][ti], T(y[idx])), batch_size=cfg['batch_size'],
                        shuffle=True, generator=torch.Generator().manual_seed(seed))
    model = Detector(name, cfg['dropout']).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    total = len(loader) * cfg['epochs']
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    if variant == 'classweight':
        cnt = np.bincount(y[tr], minlength=2)
        crit = nn.CrossEntropyLoss(weight=torch.tensor(len(tr) / (2 * cnt), dtype=torch.float32).to(DEVICE))
    else:
        crit = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    has_val = va is not None and len(va) > 0
    best_auc, best_state, best_ep, best_mv, hist = -1, None, -1, None, []
    for ep in range(cfg['epochs']):
        model.train(); tl = n = 0
        for xb, mb, yb in loader:
            xb, mb, yb = xb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                logits, _ = model(xb, mb)
            loss = crit(logits.float(), yb)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
            tl += loss.item() * len(yb); n += len(yb)
        row = dict(epoch=ep + 1, train_loss=tl / n)
        if has_val:
            pv = predict_probs(model, D['ids'][T(va)], D['mask'][T(va)]); mv = metrics_from(y[va], pv)
            row.update(val_auc=float(mv['auc']), val_acc=float(mv['acc']), val_macro_f1=float(mv['macro_f1']))
            print(f"   [{name} | {variant} | seed {seed}] epoch {ep+1}: train_loss={tl/n:.4f} val_acc={mv['acc']:.4f} val_auc={mv['auc']:.4f}")
            if mv['auc'] > best_auc:
                best_auc, best_ep, best_mv = mv['auc'], ep + 1, mv
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            print(f"   [{name} | {variant} | seed {seed}] epoch {ep+1}: train_loss={tl/n:.4f}")
        hist.append(row)
    if best_state is not None:
        model.load_state_dict(best_state)
    res = dict(model=name, variant=variant, seed=seed, best_epoch=(best_ep if has_val else cfg['epochs']),
               history=hist, **{f'info_{k}': v for k, v in info.items() if k != 'variant'})
    if best_mv is not None:
        res.update(val_auc=float(best_mv['auc']), val_acc=float(best_mv['acc']), val_macro_f1=float(best_mv['macro_f1']))
    pt = None
    if eval_test and te is not None:
        pt = predict_probs(model, D['ids'][T(te)], D['mask'][T(te)])
        res.update(metrics_from(y[te], pt))
    res['train_seconds'] = round(time.time() - t_start, 1)
    if save_path:
        torch.save(best_state if best_state is not None else {k: v.detach().cpu() for k, v in model.state_dict().items()}, save_path)
    del model, opt; torch.cuda.empty_cache()
    return res, pt, hist


## 4. Helpers: grid runner, statistics, latency, cheap baselines

In [ ]:
from glob import glob

def run_grid(df, split, models, variants, seeds, tag='main', save_for=('distilbert-base-uncased', 'smote_nn', 42)):
    tr, va, te = split
    for name in models:
        D = get_D(name, df, need_emb=('smote_nn' in variants))
        for variant in variants:
            for seed in seeds:
                key = f"{tag}_{name.replace('/', '_')}_{variant}_{seed}"
                base = f"{CFG['save_dir']}/preds/{key}"
                if os.path.exists(base + '.json'):
                    print('skip (already done):', key); continue
                sp = base.replace('/preds/', '/') + '.pt' if (name, variant, seed) == save_for else None
                res, pt, hist = run_experiment(name, variant, seed, D, split, CFG, save_path=sp)
                res['tag'] = tag
                np.savez(base + '.npz', probs=pt, labels=D['y'][te], te_idx=te)
                json.dump(res, open(base + '.json', 'w'))
                print(f"=> {key}: test acc={res['acc']:.4f} auc={res['auc']:.4f} "
                      f"human_recall={res['human_recall']:.4f} ai_recall={res['ai_recall']:.4f}")

def summarize(tag='main'):
    R = pd.DataFrame([json.load(open(f)) for f in glob(f"{CFG['save_dir']}/preds/{tag}_*.json")])
    cols = ['acc', 'macro_f1', 'auc', 'bal_acc', 'human_recall', 'ai_recall', 'ai_precision']
    g = R.groupby(['model', 'variant'])
    mean, std = g[cols].mean() * 100, g[cols].std(ddof=1).fillna(0) * 100
    out = mean.round(2).astype(str) + ' ± ' + std.round(2).astype(str)
    out.insert(0, 'n_seeds', g.size())
    out.to_csv(f"{CFG['save_dir']}/summary_{tag}.csv")
    return out, R

def pairwise_stats(tag='main', model='distilbert-base-uncased', seed=42, ref='smote_nn'):
    """Bootstrap 95% CIs and exact McNemar tests (same test set, same seed)."""
    P = {}
    for v in CFG['variants']:
        f = f"{CFG['save_dir']}/preds/{tag}_{model.replace('/', '_')}_{v}_{seed}.npz"
        if os.path.exists(f): P[v] = np.load(f)
    for v, z in P.items():
        y, p = z['labels'], z['probs']
        a = bootstrap_ci(y, p, lambda yy, pp: ((pp >= .5) == yy).mean())
        u = bootstrap_ci(y, p, roc_auc_score)
        print(f"{v:12s} acc 95% CI [{a[0]*100:.2f}, {a[1]*100:.2f}]   AUC 95% CI [{u[0]*100:.2f}, {u[1]*100:.2f}]")
    if ref in P:
        for v, z in P.items():
            if v == ref: continue
            b, c, pv = mcnemar_exact(P[ref]['labels'], (P[ref]['probs'] >= .5).astype(int), (z['probs'] >= .5).astype(int))
            print(f"McNemar {ref} vs {v}: only-{ref}-correct={b}, only-{v}-correct={c}, exact p={pv:.4f}")

def report_run(key, thresholds=(0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99)):
    z = np.load(f"{CFG['save_dir']}/preds/{key}.npz"); y, p = z['labels'], z['probs']
    m = metrics_from(y, p)
    print(f"RAW CONFUSION COUNTS (rows = actual, cols = predicted), n={len(y)}")
    print(f"             pred Human   pred AI\n  actual Human  {m['tn']:8d}  {m['fp']:8d}\n  actual AI     {m['fn']:8d}  {m['tp']:8d}")
    print(f"Human: precision={m['human_precision']:.4f} recall={m['human_recall']:.4f} | "
          f"AI: precision={m['ai_precision']:.4f} recall={m['ai_recall']:.4f}")
    print('\nDecision-threshold sweep (P(AI) >= thr  => flagged as AI):')
    rows = [dict(thr=t, human_recall=metrics_from(y, p, t)['human_recall'], ai_recall=metrics_from(y, p, t)['ai_recall'],
                 ai_precision=metrics_from(y, p, t)['ai_precision']) for t in thresholds]
    print(pd.DataFrame(rows).round(4).to_string(index=False))

def cross_source(name, train_src, test_src, variants, seed=42):
    S, y, g = df['source'].values, df['label'].values, df['group'].values
    A, B = np.where(S == train_src)[0], np.where(S == test_src)[0]
    sg = StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(iter(sg.split(A, y[A], g[A])))
    split = (A[tr_rel], A[va_rel], B)
    run_grid(df, split, [name], variants, [seed], tag=f'x_{train_src}_to_{test_src}')

@torch.no_grad()
def benchmark_latency(texts, name='distilbert-base-uncased', batch_sizes=(1, 16, 32), warmup=20, reps=200,
                      device=None, fp16=False, padding='max_length'):
    dev = device or DEVICE
    tok = AutoTokenizer.from_pretrained(name)
    model = Detector(name).to(dev).eval()      # weights do not change latency, so a random head is fine
    use16 = fp16 and dev.type == 'cuda'
    rng = np.random.RandomState(0); rows = []
    reps = reps if dev.type == 'cuda' else min(reps, 40)
    for bs in batch_sizes:
        tot, tk, md = [], [], []
        for i in range(warmup + reps):
            batch = list(rng.choice(texts, bs))
            t0 = time.perf_counter()
            enc = tok(batch, max_length=CFG['max_len'], padding=padding, truncation=True, return_tensors='pt')
            t1 = time.perf_counter()
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use16):
                model(enc['input_ids'].to(dev), enc['attention_mask'].to(dev))
            if dev.type == 'cuda': torch.cuda.synchronize()
            t2 = time.perf_counter()
            if i >= warmup: tk.append(t1 - t0); md.append(t2 - t1); tot.append(t2 - t0)
        tot, tk, md = np.array(tot) * 1e3, np.array(tk) * 1e3, np.array(md) * 1e3
        rows.append(dict(device=(torch.cuda.get_device_name(0) if dev.type == 'cuda' else 'CPU'), precision='FP16' if use16 else 'FP32',
                         batch=bs, reps=reps, tokenize_ms=tk.mean(), model_ms=md.mean(), total_ms_per_batch=tot.mean(),
                         p50=np.percentile(tot, 50), p95=np.percentile(tot, 95), p99=np.percentile(tot, 99),
                         ms_per_doc=tot.mean() / bs, docs_per_sec=1000 * bs / tot.mean()))
    del model; torch.cuda.empty_cache()
    return pd.DataFrame(rows).round(2)

def cheap_baselines(df, emb, split, seeds):
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.feature_extraction.text import TfidfVectorizer
    from imblearn.over_sampling import SMOTE as _SMOTE, RandomOverSampler
    tr, va, te = split; y = df['label'].values; rows = []
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200000)
    Xtr, Xte = vec.fit_transform(df['text'].iloc[tr]), vec.transform(df['text'].iloc[te])
    for cw in [None, 'balanced']:
        clf = LogisticRegression(C=10, max_iter=3000, class_weight=cw).fit(Xtr, y[tr])
        rows.append(dict(method=f"TF-IDF + LR ({'class-weighted' if cw else 'plain'})", seed=0,
                         **metrics_from(y[te], clf.predict_proba(Xte)[:, 1])))
    if emb is not None:
        sc = StandardScaler().fit(emb[tr]); Etr, Ete = sc.transform(emb[tr]), sc.transform(emb[te])
        for seed in seeds:
            for nm, smp, cw in [('plain', None, None), ('class-weighted', None, 'balanced'),
                                ('random oversampling', RandomOverSampler(random_state=seed), None),
                                ('SMOTE', _SMOTE(k_neighbors=5, random_state=seed), None)]:
                X, yy = (Etr, y[tr]) if smp is None else smp.fit_resample(Etr, y[tr])
                clf = LogisticRegression(max_iter=3000, class_weight=cw).fit(X, yy)
                rows.append(dict(method=f'Frozen-encoder embeddings + LR head ({nm})', seed=seed,
                                 **metrics_from(y[te], clf.predict_proba(Ete)[:, 1])))
    R = pd.DataFrame(rows); cols = ['acc', 'macro_f1', 'auc', 'bal_acc', 'human_recall', 'ai_recall']
    g = R.groupby('method', sort=False)
    mean, std = g[cols].mean() * 100, g[cols].std(ddof=1).fillna(0) * 100
    return mean.round(2).astype(str) + ' ± ' + std.round(2).astype(str)

# ------------------------------------------------------------------ hyper-parameter search (validation split only)
HP_CONFIGS = [
    dict(lr=2e-5, dropout=0.3, batch_size=16),
    dict(lr=3e-5, dropout=0.3, batch_size=16),
    dict(lr=2e-5, dropout=0.4, batch_size=16),
    dict(lr=3e-5, dropout=0.2, batch_size=32),
    dict(lr=1e-5, dropout=0.3, batch_size=16),
]

def hparam_search(df, split, name='distilbert-base-uncased', seeds=(42, 43), variant='none', epochs=3):
    """Tunes on train/val ONLY (test split is never touched). Uses the un-balanced baseline so tuning does not favour SMOTE."""
    tr, va, _ = split; D = get_D(name, df); rows = []
    for i, hp in enumerate(HP_CONFIGS):
        for seed in seeds:
            f = f"{CFG['save_dir']}/preds/hp_{i}_{seed}.json"
            if os.path.exists(f):
                rows.append(json.load(open(f))); print('skip (already done): hp', i, seed); continue
            cfg = {**CFG, **hp, 'epochs': epochs}
            res, _, _ = run_experiment(name, variant, seed, D, (tr, va, None), cfg, eval_test=False)
            res.update(cfg_id=i, **hp); json.dump(res, open(f, 'w')); rows.append(res)
    H = pd.DataFrame(rows)
    g = H.groupby('cfg_id')
    agg = pd.DataFrame({'lr': g['lr'].first(), 'dropout': g['dropout'].first(), 'batch_size': g['batch_size'].first(),
                        'val_acc': g['val_acc'].mean(), 'val_acc_sd': g['val_acc'].std(ddof=1).fillna(0),
                        'val_f1': g['val_macro_f1'].mean(), 'val_f1_sd': g['val_macro_f1'].std(ddof=1).fillna(0),
                        'val_auc': g['val_auc'].mean(), 'val_auc_sd': g['val_auc'].std(ddof=1).fillna(0),
                        'n_seeds': g.size()})
    best = int(agg['val_auc'].idxmax())
    agg['selected'] = [i == best for i in agg.index]
    agg.to_csv(f"{CFG['save_dir']}/hparam_search.csv")
    vcols = [c for c in agg.columns if c.startswith('val_')]
    agg[vcols] = agg[vcols].round(4)          # keep lr unrounded (1e-5 would become 0)
    return agg, {k: HP_CONFIGS[best][k] for k in ('lr', 'dropout', 'batch_size')}

# ------------------------------------------------------------------ 5-fold group-aware CV on train+val
def run_cv(df, split, variants, k=5, seed=42):
    """CV on the development data only (test split excluded). Fixed number of epochs, NO checkpoint selection on the held-out fold."""
    tr, va, _ = split; dev = np.concatenate([tr, va]); y, g = df['label'].values, df['group'].values
    name = CFG['models'][0]; D = get_D(name, df, need_emb=('smote_nn' in variants))
    sg = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
    for fold, (a, b) in enumerate(sg.split(dev, y[dev], g[dev])):
        for variant in variants:
            base = f"{CFG['save_dir']}/preds/cv_{variant}_{fold}"
            if os.path.exists(base + '.json'):
                print('skip (already done): cv', variant, fold); continue
            res, pt, _ = run_experiment(name, variant, seed + fold, D, (dev[a], None, dev[b]), CFG)
            res.update(tag='cv', fold=fold)
            np.savez(base + '.npz', probs=pt, labels=y[dev[b]]); json.dump(res, open(base + '.json', 'w'))
            print(f"=> cv fold {fold} {variant}: acc={res['acc']:.4f} auc={res['auc']:.4f}")

def model_size(name='distilbert-base-uncased'):
    m = Detector(name); n = sum(p.numel() for p in m.parameters())
    print(f"{name}: {n:,} parameters ({n/1e6:.1f}M), FP32 weights ~{n*4/1e6:.0f} MB, FP16 ~{n*2/1e6:.0f} MB")
    del m

# ------------------------------------------------------------------ figures (IEEE-sized, 300 dpi)
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'legend.fontsize': 7,
                     'xtick.labelsize': 7, 'ytick.labelsize': 7, 'figure.dpi': 110})
VAR_LABEL = {'none': 'No balancing', 'classweight': 'Class-weighted loss', 'ros': 'Random oversampling', 'smote_nn': 'Latent SMOTE'}
os.makedirs(f"{CFG['save_dir']}/figures", exist_ok=True)

def _save(fig, name):
    fig.savefig(f"{CFG['save_dir']}/figures/{name}.png", dpi=300, bbox_inches='tight'); plt.show()

def _main_runs(tag='main'):
    R = pd.DataFrame([json.load(open(f)) for f in glob(f"{CFG['save_dir']}/preds/{tag}_*.json")])
    return R[R['model'] == CFG['models'][0]]

def fig_hparam(agg):
    cfg_lbl = [f"{r.lr*1e5:g}e-5\np={r.dropout}\nB={int(r.batch_size)}" for r in agg.itertuples()]
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.4)); x = np.arange(len(agg))
    for ax, (m, title) in zip(axes, [('val_acc', 'Validation accuracy (%)'), ('val_f1', 'Validation macro-F1 (%)'), ('val_auc', 'Validation AUC (%)')]):
        mean, sd = agg[m].values * 100, agg[m + '_sd'].values * 100
        bars = ax.bar(x, mean, yerr=sd, capsize=2, color='#7aa6d6', edgecolor='black', linewidth=0.5)
        for b, sel in zip(bars, agg['selected']):
            if sel: b.set_hatch('///'); b.set_facecolor('#2f6db3')
        ax.set_xticks(x); ax.set_xticklabels(cfg_lbl); ax.set_title(title)
        ax.set_ylim(max(0, (mean - sd).min() - 1.5), min(100, (mean + sd).max() + 1)); ax.grid(axis='y', alpha=.3)
    n = int(agg['n_seeds'].iloc[0])
    fig.suptitle(f"Hyper-parameter search (train/val only; mean ± SD over {n} seed(s); hatched = selected; labels: lr / dropout p1 / batch)", fontsize=8, y=1.04)
    fig.tight_layout(); _save(fig, 'fig_hparam_search')

def fig_curves(R, variants=None):
    variants = variants or [v for v in CFG['variants'] if v in set(R['variant'])]
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.4))
    for v, c in zip(variants, ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']):
        sub = R[R.variant == v]
        for ax, key in zip(axes, ['train_loss', 'val_auc', 'val_acc']):
            H = np.array([[h[key] for h in hist] for hist in sub['history']]); ep = np.arange(1, H.shape[1] + 1)
            ax.plot(ep, H.mean(0), marker='o', ms=3, color=c, label=VAR_LABEL[v])
            ax.fill_between(ep, H.mean(0) - H.std(0), H.mean(0) + H.std(0), color=c, alpha=.15)
    for ax, t in zip(axes, ['Training loss', 'Validation AUC', 'Validation accuracy']):
        ax.set_title(t); ax.set_xlabel('Epoch'); ax.grid(alpha=.3)
    axes[0].set_yscale('log'); axes[0].legend(); fig.tight_layout(); _save(fig, 'fig_learning_curves')

def fig_final(key):
    from sklearn.metrics import roc_curve
    z = np.load(f"{CFG['save_dir']}/preds/{key}.npz"); y, p = z['labels'], z['probs']; m = metrics_from(y, p)
    cm = np.array([[m['tn'], m['fp']], [m['fn'], m['tp']]]); pct = cm / cm.sum(1, keepdims=True) * 100
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))
    axes[0].imshow(pct, cmap='Blues', vmin=0, vmax=100)
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, f"{cm[i, j]:,}\n({pct[i, j]:.1f}%)", ha='center', va='center', color='white' if pct[i, j] > 50 else 'black')
    axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['Human', 'AI']); axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['Human', 'AI'])
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual'); axes[0].set_title(f'Test confusion matrix (n={len(y):,}; counts and row %)')
    fpr, tpr, _ = roc_curve(y, p); axes[1].plot(fpr, tpr, color='#9C27B0', lw=2, label=f"AUC = {m['auc']:.4f}")
    axes[1].plot([0, 1], [0, 1], 'k--', lw=.8); axes[1].set_xlabel('False-positive rate'); axes[1].set_ylabel('True-positive rate')
    axes[1].set_title('ROC curve (test)'); axes[1].legend(loc='lower right'); axes[1].grid(alpha=.3)
    fig.tight_layout(); _save(fig, 'fig_final_confusion_roc')

def fig_ablation(R):
    variants = [v for v in CFG['variants'] if v in set(R['variant'])]; g = R.groupby('variant')
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6))
    for ax, mets, title in [(axes[0], [('acc', 'Accuracy'), ('macro_f1', 'Macro-F1'), ('auc', 'AUC')], 'Overall test metrics'),
                            (axes[1], [('human_recall', 'Human recall'), ('ai_recall', 'AI recall'), ('ai_precision', 'AI precision')], 'Per-class test metrics')]:
        w = 0.8 / len(mets); x = np.arange(len(variants)); lo = 100
        for k, (m, lbl) in enumerate(mets):
            mean = np.array([g.get_group(v)[m].mean() for v in variants]) * 100
            sd = np.array([g.get_group(v)[m].std(ddof=1) if len(g.get_group(v)) > 1 else 0 for v in variants]) * 100
            ax.bar(x + k * w, mean, w, yerr=np.nan_to_num(sd), capsize=2, label=lbl, edgecolor='black', linewidth=.4); lo = min(lo, (mean - np.nan_to_num(sd)).min())
        ax.set_xticks(x + w * (len(mets) - 1) / 2); ax.set_xticklabels([VAR_LABEL[v].replace(' ', '\n', 1) for v in variants])
        ax.set_ylim(max(0, lo - 2), 100.5); ax.set_title(title + ' (%, mean ± SD over seeds)'); ax.legend(loc='lower right'); ax.grid(axis='y', alpha=.3)
    fig.tight_layout(); _save(fig, 'fig_ablation')

# ------------------------------------------------------------------ results pack
def export_pack():
    import zipfile
    d = CFG['save_dir']
    env = (f"torch {torch.__version__} | device {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}\n"
           f"CFG = {json.dumps(CFG, indent=1)}")
    open(f"{d}/RESULTS_FOR_PAPER.txt", 'w').write(env + '\n' + '\n'.join(REPORT))
    zpath = os.path.join(os.path.dirname(d.rstrip('/')), 'deepfakeguard_results_pack.zip')
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        for root, _, files_ in os.walk(d):
            for f in files_:
                if f.endswith(('.txt', '.csv', '.json', '.png', '.npz')):
                    z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), d))
    print('Wrote', zpath, f"({os.path.getsize(zpath)/1e6:.1f} MB)  |  main text file: {d}/RESULTS_FOR_PAPER.txt")
    try:
        from google.colab import files; files.download(zpath)
    except Exception:
        pass


## Step A · Data audit, split and leakage analysis

In [ ]:
df, prep_log = load_data(CFG)
split = make_splits(df, seed=42)
RELEASED = '/content/split_assignments.csv'      # upload the released file for EXACT reproduction (re-deriving can differ across scikit-learn versions)
if os.path.exists(RELEASED):
    A = pd.read_csv(RELEASED, index_col=0)
    assert len(A) == len(df) and (A['label'].values == df['label'].values).all() and (A['source'].values == df['source'].values).all()
    split = tuple(np.where(A['split'] == nm)[0] for nm in ('train', 'val', 'test')); print('Using the released split from', RELEASED)
tr, va, te = split

def audit():
    print(prep_log.to_string(index=False)); print()          # class counts after each preprocessing step
    print(df.groupby(['source', 'label']).size().unstack(fill_value=0).rename(columns={0: 'human', 1: 'AI'})); print()
    for nm, ix in [('train', tr), ('val', va), ('test', te)]:
        print(f"{nm:5s}: {len(ix):6,d} samples | AI share = {df['label'].iloc[ix].mean()*100:.1f}%")
    print("Group overlap between splits: none (asserted in make_splits)\n")
    leakage_report(df, tr, te)
capture_section('DATA AUDIT, SPLITS, LEAKAGE', audit)

assign = df[['source', 'label', 'group']].copy(); assign['split'] = 'train'
assign.loc[va, 'split'] = 'val'; assign.loc[te, 'split'] = 'test'
assign.to_csv(f"{CFG['save_dir']}/split_assignments.csv")

## Step B · Cheap baselines: TF-IDF and SMOTE on frozen embeddings (minutes)

In [ ]:
D0 = get_D('distilbert-base-uncased', df, need_emb=True)
cheap = capture_section('CHEAP BASELINES / FROZEN-EMBEDDING ABLATION (test, mean ± std %)',
                        lambda: cheap_baselines(df, D0['emb'], split, CFG['seeds']))
cheap.to_csv(f"{CFG['save_dir']}/cheap_baselines.csv")

## Step C · Hyper-parameter search on train/val only (~30–60 min)
Same 5 configurations as before, but tuned on the **validation split only** (test never touched), with the un-balanced baseline so tuning does not favour SMOTE. If you are short on time use `seeds=(42,)` (then Fig. 2 has no error bars and the paper must say it is a single run).

In [ ]:
hp_table, best_hp = hparam_search(df, split, seeds=(42, 43))
capture_section('HYPER-PARAMETER SEARCH (validation split only; mean ± SD over seeds)', lambda: hp_table)
CFG.update(best_hp); REPORT.append(f"Selected hyper-parameters: {best_hp}")
print('Selected:', best_hp)

## Step D · Main fine-tuning grid
4 variants × 3 seeds × 4 epochs (~1.5–2.5 h on a T4; every run is saved when it finishes, so re-running skips finished runs).
`smote_nn` = SMOTE on frozen-DistilBERT [CLS] embeddings, each synthetic vector mapped to its nearest real **minority** text; the whole network is then fine-tuned end-to-end.

In [ ]:
run_grid(df, split, CFG['models'], CFG['variants'], CFG['seeds'])

## Step E · Results tables, statistics, confusion counts, thresholds

In [ ]:
tab, R = summarize('main')
capture_section('MAIN RESULTS: test set, mean ± std over seeds (%) [test scored once; checkpoint chosen on validation split]', lambda: tab)
capture_section('BOOTSTRAP 95% CIs AND EXACT McNEMAR TESTS (seed 42)', lambda: pairwise_stats('main'))
capture_section('HEADLINE RUN: RAW CONFUSION COUNTS + THRESHOLD SWEEP', lambda: report_run('main_distilbert-base-uncased_smote_nn_42'))
info_cols = [c for c in ['variant', 'seed', 'best_epoch', 'train_seconds', 'info_n_extra', 'info_unique_source_texts',
                          'info_pct_synthetic_that_duplicate_an_already_used_text'] if c in R]
capture_section('WHAT THE SMOTE STEP ACTUALLY PRODUCES', lambda: R[R.variant == 'smote_nn'][info_cols])

## Step F · 5-fold group-aware CV on train+val (~1–1.5 h)
Replaces the old Table II. Epochs are fixed and **no checkpoint is chosen on the held-out fold**, so these numbers are not optimistically selected. The test split is excluded. If short on time set `CFG['cv_variants'] = ['smote_nn']` first.

In [ ]:
run_cv(df, split, CFG['cv_variants'])
cv_tab, _ = summarize('cv')
capture_section('5-FOLD GROUP-AWARE CV on train+val (mean ± std over folds, %; fixed epochs, no selection)', lambda: cv_tab)

## Step G · Cross-source generalisation (single seed)

In [ ]:
cross_source('distilbert-base-uncased', 'aigtxt', 'reddit', ['none'])
cross_source('distilbert-base-uncased', 'reddit', 'aigtxt', ['none', 'classweight', 'smote_nn'])
for t in ['x_aigtxt_to_reddit', 'x_reddit_to_aigtxt']:
    out, _ = summarize(t)
    capture_section(f'CROSS-SOURCE: {t}  (test = the other dataset, %)', lambda out=out: out)
# AIGTxt is perfectly balanced, so SMOTE / ROS are no-ops there; all imbalance comes from Reddit.

## Step H · Model size and latency benchmark

In [ ]:
capture_section('MODEL SIZE', model_size)
texts = list(df['text'].iloc[te])
lat = pd.concat([benchmark_latency(texts, fp16=False),
                 benchmark_latency(texts, fp16=True),
                 benchmark_latency(texts, device=torch.device('cpu'))], ignore_index=True)
lat.to_csv(f"{CFG['save_dir']}/latency.csv", index=False)
capture_section(f"LATENCY (torch {torch.__version__}; max_len={CFG['max_len']}, padding=max_length, warm-up 20, reps 200 GPU / 40 CPU)", lambda: lat)

## Step I · Figures for the paper (saved at 300 dpi)

In [ ]:
fig_hparam(hp_table)
R = _main_runs()
fig_curves(R)
fig_final('main_distilbert-base-uncased_smote_nn_42')
fig_ablation(R)

## Step J · Export everything
Creates `RESULTS_FOR_PAPER.txt` plus a zip with all tables, per-run JSONs, predictions and figures (model checkpoints excluded). **Send me `RESULTS_FOR_PAPER.txt` (paste or upload) and the zip.**

In [ ]:
export_pack()

## Optional · Paraphrase / adversarial spot-check
Take ~200 AI-labelled **test** texts, paraphrase them with any LLM, paste the results into `paraphrased`, and compare detection rates. Label it a *pilot* in the paper.

In [ ]:
def predict_texts(texts, ckpt, name='distilbert-base-uncased'):
    tok = AutoTokenizer.from_pretrained(name); m = Detector(name).to(DEVICE)
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE)); m.eval()
    enc = tok(list(texts), max_length=CFG['max_len'], padding='max_length', truncation=True, return_tensors='pt')
    return predict_probs(m, enc['input_ids'], enc['attention_mask'])

ckpt = f"{CFG['save_dir']}/main_distilbert-base-uncased_smote_nn_42.pt"
orig_ai = df[df.index.isin(te) & (df.label == 1)]['text'].sample(200, random_state=0).tolist()
print("Detection rate on original AI texts:", (predict_texts(orig_ai, ckpt) >= .5).mean())
# paraphrased = [...]   # paste LLM paraphrases of orig_ai here, same order
# print("Detection rate on paraphrased AI texts:", (predict_texts(paraphrased, ckpt) >= .5).mean())